# ASNA Actuarial Case Competition · Mortality prediction

Our four-person team placed in the top 16. This notebook reconstructs the model behind our selected Kaggle submission. It estimates five-year mortality probabilities using 10,000 training records and generates predictions for 10,000 test records. The submitted file scored **0.07361** on the private leaderboard and **0.08355** on the public leaderboard (log loss, lower is better).

To run it, put the competition’s `train.csv` and `test.csv` beside this notebook and run the cells in order. Tested with pandas 2.3.3 and scikit-learn 1.9.0.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

data_dir = Path(".")  # Folder containing the Kaggle train.csv and test.csv
train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")

## Model

An age-by-smoking interaction lets the relationship between age and log odds of death differ by smoking status. The model also uses BMI, health status, GLP-1 use, and duration of use. Missing durations are filled with zero, as in the competition workflow. L1 regularization shrinks coefficients during fitting.

In [ ]:
def model_features(frame):
    features = frame[["Health Status", "Smoker Status", "Age", "BMI", "GLP-1",
                      "GLP-1 Duration in Years"]].copy()
    features["GLP-1 Duration in Years"] = features["GLP-1 Duration in Years"].fillna(0)
    smoker = features["Smoker Status"].map({"Yes": 1, "No": 0})
    if smoker.isna().any():
        raise ValueError("Expected Yes or No in Smoker Status")
    features["Age_x_Smoker"] = features["Age"] * smoker
    return features

categorical = ["Health Status", "Smoker Status"]
numeric = ["Age", "BMI", "GLP-1", "GLP-1 Duration in Years", "Age_x_Smoker"]
preprocess = ColumnTransformer([
    ("categories", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("numeric", "passthrough", numeric),
])
model = make_pipeline(preprocess, LogisticRegression(
    l1_ratio=1.0, solver="liblinear", C=1, max_iter=20000, random_state=42
))
model.fit(model_features(train), train["Death Flag"])
probability = model.predict_proba(model_features(test))[:, 1]

## Kaggle submission

Each row pairs a test ID with its estimated probability of death.

In [ ]:
submission = pd.DataFrame({"ID": test["ID"], "Death_Prob": probability})
submission.to_csv("ASNA_Age_Smoker_L1_Submission.csv", index=False)